# Liu2024 Paper-Mimic CSP/FBCSP Baseline from Original `.mat` Trial Files

This notebook is meant to reproduce the **style of the Liu2024 Technical Validation baseline**, not the S-JEPA/MOABB downstream pipeline.

Key idea:
- Load the original Figshare **source `.mat` files** containing `rawdata` and `labels`.
- Treat each subject as 40 already-epoched 8-second trials at 500 Hz.
- Use the motor-imagery interval inside each trial, default **2–6 seconds**.
- Apply paper-like preprocessing for classical decoding: mean removal, referencing, 8–30 Hz Butterworth filtering, then CSP+LDA and FBCSP+SVM.
- Use repeated 60/40 stratified train/test splits because the paper text says 60% train / 40% test and 10-fold validation.

Run this notebook if you want to compare against the reported CSP+LDA ≈ 55.57% and FBCSP+SVM ≈ 57.57% baseline.


# 1. Setup

In [1]:
import os
import sys
import json
import zipfile
import shutil
import platform
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import io as scipy_io
from scipy.signal import butter, sosfiltfilt

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, balanced_accuracy_score, cohen_kappa_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from mne.decoding import CSP
except Exception as exc:
    raise RuntimeError('This notebook needs MNE installed: pip install mne') from exc

print('Runtime Environment:')
print(f'  Python:   {sys.version}')
print(f'  Platform: {platform.platform()}')
print(f'  Workdir:  {Path.cwd()}')


Runtime Environment:
  Python:   3.11.15 (main, Apr  9 2026, 01:18:52) [Clang 21.0.0 (clang-2100.0.123.102)]
  Platform: macOS-26.2-arm64-arm-64bit
  Workdir:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src


# 2. Paths and constants

Edit only the paths below if needed. The notebook can use either:

1. an already extracted `sourcedata` folder, or  
2. a local `sourcedata.zip`, or  
3. automatic Figshare API download if internet is available.


In [2]:
# ---- Dataset paths ----
WORKING_DIR = Path.cwd()
DATA_ROOT = WORKING_DIR / 'liu2024_figshare'
SOURCE_ZIP_PATH = DATA_ROOT / 'sourcedata.zip'
SOURCE_EXTRACT_DIR = DATA_ROOT / 'sourcedata'

# Optional: if you already downloaded/extracted elsewhere, set this to that folder.
# Expected content: sub-01/...mat, sub-02/...mat, etc. or nested folders containing those .mat files.
LOCAL_SOURCE_DIR_OVERRIDE = None

# ---- Paper-mimic constants ----
FS = 500
N_SUBJECTS = 50
RANDOM_STATE = 2026
N_SPLITS = 10
TEST_SIZE = 0.40

# The full source trial is 8 seconds = 4000 samples.
# The MI video/cue interval is 2s to 6s in the full trial.
# This corresponds to the 4s motor-imagery segment used by MOABB interval=(2, 6).
MI_WINDOW_S = (2.0, 6.0)

# EEG channel handling.
# Liu paper source data says rawdata contains 33 channels: 30 EEG, 2 EOG, 1 marker.
# We keep the first 30, then automatically remove near-zero variance channels if present.
EEG_CHANNEL_INDICES = list(range(30))
DROP_NEAR_ZERO_VARIANCE_CHANNELS = True
NEAR_ZERO_VAR_EPS = 1e-18
APPLY_AVERAGE_REFERENCE = True

# Paper-like frequency ranges.
CSP_BAND = (8.0, 30.0)
FILTER_ORDER = 2

# FBCSP paper method reports FBCSP + SVM. These bands are a practical filter-bank variant.
FBCSP_BANDS = [(8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 30)]

# CSP components. With tiny 40-trial subjects, keep this modest.
N_CSP_COMPONENTS = 4

ARTIFACT_DIR = WORKING_DIR / 'artifacts' / 'liu2024-paper-mimic-csp-fbcsp' / datetime.now().strftime('%Y%m%d_%H%M%S')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / 'run.log'
print(f'Artifacts: {ARTIFACT_DIR}')


Artifacts: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-paper-mimic-csp-fbcsp/20260518_132806


# 3. Logger

In [3]:
_log_handle = open(LOG_PATH, 'w', buffering=1)

def log(msg=''):
    text = str(msg)
    print(text)
    _log_handle.write(text + '\n')

log('Liu2024 paper-mimic CSP/FBCSP run')
log(f'Artifacts: {ARTIFACT_DIR}')


Liu2024 paper-mimic CSP/FBCSP run
Artifacts: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-paper-mimic-csp-fbcsp/20260518_132806


# 4. Get original source `.mat` files

The paper's baseline is easiest to mimic from `sourcedata.zip`, because those files contain already epoched 8-second trials: `rawdata` and `labels`.


In [ ]:
FIGSHARE_ARTICLE_ID = 21679035
FIGSHARE_VERSION = 5

def find_source_mat_files(root: Path):
    root = Path(root)
    return sorted(root.rglob('*.mat'))

def download_figshare_file_by_name(target_name: str, output_path: Path):
    import requests
    api_url = f'https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE_ID}/versions/{FIGSHARE_VERSION}'
    log(f'Querying Figshare API: {api_url}')
    meta = requests.get(api_url, timeout=30).json()
    files = meta.get('files', [])
    available = [(f.get('name'), f.get('size'), f.get('download_url')) for f in files]
    log('Available Figshare files:')
    for name, size, _ in available:
        log(f'  {name} ({size} bytes)')
    match = None
    for f in files:
        if f.get('name') == target_name:
            match = f
            break
    if match is None:
        raise FileNotFoundError(f'Could not find {target_name} in Figshare files. Available: {available}')
    url = match['download_url']
    output_path.parent.mkdir(parents=True, exist_ok=True)
    log(f'Downloading {target_name} to {output_path}')
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(output_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    log(f'Downloaded {output_path} ({output_path.stat().st_size} bytes)')

def ensure_source_dir():
    if LOCAL_SOURCE_DIR_OVERRIDE is not None:
        p = Path(LOCAL_SOURCE_DIR_OVERRIDE)
        mats = find_source_mat_files(p)
        if not mats:
            raise FileNotFoundError(f'No .mat files found in override folder: {p}')
        log(f'Using LOCAL_SOURCE_DIR_OVERRIDE with {len(mats)} .mat files: {p}')
        return p

    if SOURCE_EXTRACT_DIR.exists() and find_source_mat_files(SOURCE_EXTRACT_DIR):
        log(f'Using existing extracted source dir: {SOURCE_EXTRACT_DIR}')
        return SOURCE_EXTRACT_DIR

    if not SOURCE_ZIP_PATH.exists():
        log('sourcedata.zip not found locally. Attempting Figshare download.')
        download_figshare_file_by_name('sourcedata.zip', SOURCE_ZIP_PATH)

    log(f'Extracting {SOURCE_ZIP_PATH} to {SOURCE_EXTRACT_DIR}')
    SOURCE_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SOURCE_ZIP_PATH, 'r') as zf:
        zf.extractall(SOURCE_EXTRACT_DIR)
    mats = find_source_mat_files(SOURCE_EXTRACT_DIR)
    log(f'Found {len(mats)} .mat files after extraction.')
    return SOURCE_EXTRACT_DIR

SOURCE_DIR = ensure_source_dir()
MAT_FILES = find_source_mat_files(SOURCE_DIR)
log(f'Total .mat files found: {len(MAT_FILES)}')
for p in MAT_FILES[:5]:
    log(f'  preview: {p}')


sourcedata.zip not found locally. Attempting Figshare download.
Querying Figshare API: https://api.figshare.com/v2/articles/21679035/versions/5
Available Figshare files:
  dataset_description.json (139 bytes)
  edffile.zip (462843633 bytes)
  participants.json (2389 bytes)
  sourcedata.zip (1874824439 bytes)
  stimuli.zip (3174875 bytes)
  task-motor-imagery_channels.tsv (709 bytes)
  task-motor-imagery_coordsystem.json (66 bytes)
  task-motor-imagery_eeg.json (569 bytes)
  task-motor-imagery_electrodes.tsv (761 bytes)
  task-motor-imagery_events.json (806 bytes)
  task-motor-imagery_events.tsv (3732 bytes)
  README.md (515 bytes)
  code.zip (20811 bytes)
  participants.tsv (3557 bytes)


# 5. Load Liu source trials

In [ ]:
def subject_id_from_path(path: Path):
    import re
    s = str(path)
    m = re.search(r'sub[-_ ]?(\d{1,2})', s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.search(r'(?:subject|subj)[-_ ]?(\d{1,2})', s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r'\d+', path.stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f'Cannot infer subject id from {path}')

def load_subject_mat(path: Path):
    mat = scipy_io.loadmat(path, squeeze_me=True, struct_as_record=False)
    if 'rawdata' not in mat or 'labels' not in mat:
        keys = sorted(k for k in mat.keys() if not k.startswith('__'))
        raise KeyError(f'{path} does not contain rawdata/labels. Keys: {keys}')
    rawdata = np.asarray(mat['rawdata'])
    labels = np.asarray(mat['labels']).astype(int).ravel()
    # Normalize to trials x channels x samples.
    if rawdata.ndim != 3:
        raise ValueError(f'Expected rawdata 3D, got {rawdata.shape} in {path}')
    if rawdata.shape[0] != labels.shape[0]:
        # Try common MATLAB orientation permutations.
        for axes in [(2, 0, 1), (1, 0, 2), (2, 1, 0)]:
            candidate = np.transpose(rawdata, axes)
            if candidate.shape[0] == labels.shape[0]:
                rawdata = candidate
                break
    if rawdata.shape[0] != labels.shape[0]:
        raise ValueError(f'Could not align trials with labels. rawdata={rawdata.shape}, labels={labels.shape}')
    return rawdata.astype(np.float64), labels

subjects = []
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    X_raw, y_raw = load_subject_mat(p)
    subjects.append({'subject_id': sid, 'path': p, 'rawdata_shape': X_raw.shape, 'labels': y_raw})
subjects = sorted(subjects, key=lambda d: d['subject_id'])

summary = []
for item in subjects:
    labels = item['labels']
    vals, counts = np.unique(labels, return_counts=True)
    summary.append({
        'subject_id': item['subject_id'],
        'path': str(item['path']),
        'rawdata_shape': str(item['rawdata_shape']),
        'label_values': str(dict(zip(vals.tolist(), counts.tolist()))),
    })
summary_df = pd.DataFrame(summary)
display(summary_df.head())
summary_df.to_csv(ARTIFACT_DIR / 'source_mat_summary.csv', index=False)
log(summary_df.to_string(index=False))


# 6. Paper-like preprocessing helpers

In [ ]:
def bandpass_zero_phase(X, sfreq, l_freq, h_freq, order=2):
    sos = butter(order, [l_freq, h_freq], btype='bandpass', fs=sfreq, output='sos')
    return sosfiltfilt(sos, X, axis=-1)

def prepare_subject_trials(rawdata, labels, window_s=MI_WINDOW_S, band=CSP_BAND):
    # 1) Select EEG channels, excluding EOG and marker.
    X = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)
    y = labels.astype(int).ravel() - 1  # paper labels 1/2 -> sklearn 0/1

    # 2) Remove DC/baseline mean per trial/channel over full trial.
    X = X - X.mean(axis=-1, keepdims=True)

    # 3) Drop near-zero channels if present, e.g. a reference-like channel.
    kept_channels = np.arange(X.shape[1])
    if DROP_NEAR_ZERO_VARIANCE_CHANNELS:
        channel_var = X.var(axis=(0, 2))
        keep_mask = channel_var > NEAR_ZERO_VAR_EPS
        if not np.all(keep_mask):
            kept_channels = kept_channels[keep_mask]
            X = X[:, keep_mask, :]

    # 4) Average reference across retained EEG channels.
    if APPLY_AVERAGE_REFERENCE:
        X = X - X.mean(axis=1, keepdims=True)

    # 5) Bandpass to CSP band.
    X = bandpass_zero_phase(X, FS, band[0], band[1], FILTER_ORDER)

    # 6) Select the 4-second MI analysis window.
    start = int(round(window_s[0] * FS))
    stop = int(round(window_s[1] * FS))
    X = X[:, :, start:stop]
    return X, y, kept_channels

# Inspect one subject after preprocessing.
sid0 = subjects[0]['subject_id']
raw0, y0 = load_subject_mat(subjects[0]['path'])
X0, yy0, kept0 = prepare_subject_trials(raw0, y0)
log(f'Example subject {sid0}: raw={raw0.shape}, prepared={X0.shape}, labels={np.bincount(yy0)}, kept_channels={kept0.tolist()}')


# 7. Models

In [ ]:
def make_csp_lda(n_components=N_CSP_COMPONENTS):
    return Pipeline([
        ('csp', CSP(n_components=n_components, reg='ledoit_wolf', log=True, norm_trace=False)),
        ('lda', LinearDiscriminantAnalysis()),
    ])

def fbcsp_features_fit_transform(X_train, y_train, X_test, bands=FBCSP_BANDS, n_components=N_CSP_COMPONENTS):
    train_features = []
    test_features = []
    fitted = []
    for band in bands:
        Xtr_b = bandpass_zero_phase(X_train, FS, band[0], band[1], FILTER_ORDER)
        Xte_b = bandpass_zero_phase(X_test, FS, band[0], band[1], FILTER_ORDER)
        csp = CSP(n_components=n_components, reg='ledoit_wolf', log=True, norm_trace=False)
        train_features.append(csp.fit_transform(Xtr_b, y_train))
        test_features.append(csp.transform(Xte_b))
        fitted.append(csp)
    return np.concatenate(train_features, axis=1), np.concatenate(test_features, axis=1), fitted

def evaluate_fold_metrics(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'kappa': float(cohen_kappa_score(y_true, y_pred)),
        'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'sensitivity_macro': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }


# 8. Run per-subject repeated 60/40 validation

This follows the paper text more closely than 5-fold S-JEPA-style CV: every subject has 40 trials, and each split uses 24 train / 16 test trials.


In [ ]:
all_rows = []
subject_rows = []

for item in subjects:
    sid = item['subject_id']
    rawdata, labels = load_subject_mat(item['path'])
    X, y, kept_channels = prepare_subject_trials(rawdata, labels, window_s=MI_WINDOW_S, band=CSP_BAND)

    if len(np.unique(y)) != 2:
        log(f'SKIP subject {sid}: labels are not binary after loading: {np.unique(y)}')
        continue

    splitter = StratifiedShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE + sid)
    for split_idx, (train_idx, test_idx) in enumerate(splitter.split(X, y), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # CSP + LDA
        csp_lda = make_csp_lda()
        csp_lda.fit(X_train, y_train)
        pred = csp_lda.predict(X_test)
        row = {
            'subject_id': sid,
            'split_id': split_idx,
            'model_name': 'CSP_LDA',
            'n_train': int(len(train_idx)),
            'n_test': int(len(test_idx)),
            'train_class_counts': np.bincount(y_train, minlength=2).tolist(),
            'test_class_counts': np.bincount(y_test, minlength=2).tolist(),
            'n_channels_used': int(X.shape[1]),
            'n_times': int(X.shape[2]),
            'window_s': list(MI_WINDOW_S),
        }
        row.update(evaluate_fold_metrics(y_test, pred))
        all_rows.append(row)

        # FBCSP + SVM, matching the paper's method family.
        Xtr_feat, Xte_feat, _ = fbcsp_features_fit_transform(X_train, y_train, X_test)
        svm = Pipeline([
            ('scale', StandardScaler()),
            ('svm', SVC(kernel='linear', C=1.0)),
        ])
        svm.fit(Xtr_feat, y_train)
        pred = svm.predict(Xte_feat)
        row = {
            'subject_id': sid,
            'split_id': split_idx,
            'model_name': 'FBCSP_SVM',
            'n_train': int(len(train_idx)),
            'n_test': int(len(test_idx)),
            'train_class_counts': np.bincount(y_train, minlength=2).tolist(),
            'test_class_counts': np.bincount(y_test, minlength=2).tolist(),
            'n_channels_used': int(X.shape[1]),
            'n_times': int(X.shape[2]),
            'window_s': list(MI_WINDOW_S),
            'n_filter_bands': len(FBCSP_BANDS),
        }
        row.update(evaluate_fold_metrics(y_test, pred))
        all_rows.append(row)

results_df = pd.DataFrame(all_rows)
results_df.to_csv(ARTIFACT_DIR / 'cv_results.csv', index=False)
display(results_df.head())
log(f'Wrote fold results: {ARTIFACT_DIR / "cv_results.csv"}')


# 9. Summaries

In [ ]:
global_metrics = (
    results_df
    .groupby('model_name')
    .agg(
        mean_accuracy=('accuracy', 'mean'),
        std_accuracy=('accuracy', 'std'),
        mean_balanced_accuracy=('balanced_accuracy', 'mean'),
        std_balanced_accuracy=('balanced_accuracy', 'std'),
        mean_kappa=('kappa', 'mean'),
        mean_precision_macro=('precision_macro', 'mean'),
        mean_sensitivity_macro=('sensitivity_macro', 'mean'),
        n_folds=('accuracy', 'size'),
    )
    .reset_index()
)
subject_metrics = (
    results_df
    .groupby(['model_name', 'subject_id'])
    .agg(
        mean_accuracy=('accuracy', 'mean'),
        std_accuracy=('accuracy', 'std'),
        mean_balanced_accuracy=('balanced_accuracy', 'mean'),
        mean_kappa=('kappa', 'mean'),
        n_splits=('accuracy', 'size'),
    )
    .reset_index()
)

display(global_metrics)
display(subject_metrics.head())

global_metrics.to_csv(ARTIFACT_DIR / 'global_metrics.csv', index=False)
subject_metrics.to_csv(ARTIFACT_DIR / 'subject_metrics.csv', index=False)

metadata = {
    'notebook': 'liu2024_paper_mimic_csp_fbcsp_source_mat',
    'source': 'original Figshare sourcedata .mat files',
    'fs': FS,
    'mi_window_s': MI_WINDOW_S,
    'validation': {'type': 'StratifiedShuffleSplit', 'n_splits': N_SPLITS, 'test_size': TEST_SIZE, 'random_state': RANDOM_STATE},
    'csp_band': CSP_BAND,
    'fbcsp_bands': FBCSP_BANDS,
    'n_csp_components': N_CSP_COMPONENTS,
    'apply_average_reference': APPLY_AVERAGE_REFERENCE,
    'drop_near_zero_variance_channels': DROP_NEAR_ZERO_VARIANCE_CHANNELS,
}
with open(ARTIFACT_DIR / 'run_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

log('\nGlobal metrics:')
log(global_metrics.to_string(index=False))
log(f'\nArtifacts saved to: {ARTIFACT_DIR}')
_log_handle.close()


# 10. Notes for interpretation

If this notebook lands closer to the paper baseline than the MOABB/S-JEPA-style baseline, the difference is probably not CSP itself. It is probably the data object and validation protocol:

- paper baseline: original 8-second epoched `.mat` trials, 500 Hz, 2–6s MI interval, 60/40 repeated validation;
- S-JEPA-style baseline: MOABB annotations, resampled to 128 Hz, 537-sample windows, within-subject 5-fold CV.
